In [ ]:
# ── Cell 1: Colab GPU Setup ───────────────────────────────────────────────────
# Requires T4 GPU runtime: Runtime → Change runtime type → T4 GPU

import subprocess, sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/fmd'
    IN_COLAB = True
except ImportError:
    PROJECT_DIR = '/Users/aman2394/Desktop/ICWSM-2026-FMD'
    IN_COLAB = False
    print('Running locally — skipping Drive mount')

os.makedirs(f'{PROJECT_DIR}/features', exist_ok=True)
sys.path.insert(0, f'{PROJECT_DIR}/src')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'pandas', 'numpy', 'tqdm', 'torch'
], check=True)

import torch
assert torch.cuda.is_available(), (
    'No GPU detected. Runtime → Change runtime type → T4 GPU'
)
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PROJECT_DIR = {PROJECT_DIR}')

In [ ]:
# ── Cell 2: Load Data (Train Split Only) ─────────────────────────────────────
# Loads train split: original train samples + augmented False samples
# Val and test sets are held out — used only in notebook 06/07.

import sys
sys.path.insert(0, f'{PROJECT_DIR}/src')

import numpy as np
from utils.data_loader import load_combined_data
from utils.data_splitter import get_split_records

orig_records = load_combined_data(PROJECT_DIR)
AUG_PATH = f'{PROJECT_DIR}/data/augmented/augmented_train.json'

train_records, val_records, test_records = get_split_records(
    all_records=orig_records,
    augmented_path=AUG_PATH,
    project_dir=PROJECT_DIR,
)

texts  = [r['text']  for r in train_records]
labels = np.array([int(r['label']) for r in train_records], dtype=np.int32)

n_true  = (labels == 1).sum()
n_false = (labels == 0).sum()
n_aug   = sum(r.get('perturbation_type') is not None for r in train_records)

print(f'Train set  : {len(texts)} samples')
print(f'  True     : {n_true}')
print(f'  False    : {n_false}  (of which {n_aug} are augmented)')
print(f'  Balance  : {labels.mean():.2%} true')
print(f'Val set    : {len(val_records)} samples (held out)')
print(f'Test set   : {len(test_records)} samples (held out)')
print(f'\nSample text: {texts[0][:200]}')


In [ ]:
# ── Cell 3: Extract MLM Perplexity Features ───────────────────────────────────
# Model : ProsusAI/finbert (MLM head)
# Method: Token-by-token masking — each token is masked once, log-prob recorded
# Output: (N, 4) array — mean_perplexity, max_perplexity, std_perplexity,
#                         top_10pct_perplexity_ratio
# Est.  : ~2 hrs on T4 (slow — sequential token masking per sample)
#
# NOTE: Dataset-specific feature. Signals unnaturally low perplexity in
# LLM-edited text vs. genuine financial journalism.

from features.tier3_perplexity import extract_perplexity_feature_matrix

print('Extracting FinBERT MLM perplexity features (~2 hrs on T4) ...')
tier3_features = extract_perplexity_feature_matrix(
    texts=texts,
    device='cuda',
    max_length=512,
    checkpoint_path=f'{PROJECT_DIR}/feature_cache/_tier3_checkpoint.npy',
    checkpoint_every=100,
)

print(f'Tier 3 features shape : {tier3_features.shape}')  # (N, 4)
print(f'Sample row            : {tier3_features[0]}')

import numpy as np
assert not np.isnan(tier3_features).any(), 'NaN values detected in tier3_features!'
print('NaN check passed.')

In [ ]:
# ── Cell 4: Save tier3_features.npy + Distribution Stats ─────────────────────

import numpy as np
import pandas as pd

# Save
np.save(f'{PROJECT_DIR}/feature_cache/tier3_features.npy', tier3_features)
print('✅ Saved:', f'{PROJECT_DIR}/feature_cache/tier3_features.npy')
print(f'   Shape: {tier3_features.shape}')

# Distribution stats split by label
feature_names = [
    'mean_perplexity',
    'max_perplexity',
    'std_perplexity',
    'top_10pct_perplexity_ratio',
]

df_stats = pd.DataFrame(tier3_features, columns=feature_names)
df_stats['label'] = labels

print('\nDistribution by label (True=1 = original, False=0 = perturbed):')
print(df_stats.groupby('label')[feature_names].agg(['mean', 'std']).round(4))

print('\nOverall distribution:')
print(df_stats[feature_names].describe().round(4))

# Visual quick-check: mean perplexity should differ between labels
try:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for lbl, color in [(0, 'red'), (1, 'blue')]:
        subset = df_stats[df_stats['label'] == lbl]['mean_perplexity']
        axes[0].hist(subset, bins=40, alpha=0.5, color=color,
                     label=f'label={lbl} (n={len(subset)})')
    axes[0].set_title('Mean Perplexity Distribution by Label')
    axes[0].set_xlabel('mean_perplexity')
    axes[0].legend()

    for lbl, color in [(0, 'red'), (1, 'blue')]:
        subset = df_stats[df_stats['label'] == lbl]['top_10pct_perplexity_ratio']
        axes[1].hist(subset, bins=40, alpha=0.5, color=color,
                     label=f'label={lbl}')
    axes[1].set_title('Top-10% Perplexity Ratio Distribution')
    axes[1].set_xlabel('top_10pct_perplexity_ratio')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f'{PROJECT_DIR}/results/tier3_perplexity_dist.png', dpi=150)
    plt.show()
    print('Distribution plot saved.')
except Exception as e:
    print(f'Plot skipped: {e}')